In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)


In [3]:
# Question 1
count = 0
for doc in documents:
    count+=1
print(count)
# 72 files


72


In [4]:
# Question 2
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"content": 2.0},
    num_results=5
)

# Answer is:
# cohorts/2026/01-agentic-rag/lessons/14-agentic-loop.md


In [5]:
from rag_helper import RAGBase
from openai import OpenAI

openai_client = OpenAI()

ragModel = RAGBase(index, openai_client)

response = ragModel.rag(question)

print(response)

# The agentic loop keeps calling the model by putting the model call inside a `while True` loop and checking whether the response contains any `function_call` items.

# - If there are function calls, the code runs the tools, appends the tool outputs to `messages`, and loops again.
# - If there are no function calls, it breaks out of the loop and stops.

# So the stop condition is: **no function calls in the model’s response**.

The agentic loop keeps calling the model in a `while True` loop. After each model response, it checks whether there were any `function_call` items:

- If there are function calls, it runs the tool, appends the tool output to the message history, and loops again.
- If there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the latest response**.


In [6]:
ragModel.cost()
# Sent around 8000 tokens for the request

Token cost:  8013


In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

# 295 chunks in here

295


In [8]:
# RAG with chunking

chunkedIndex = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunkedIndex.fit(chunks)

search_results = chunkedIndex.search(
    question,
    boost_dict={"content": 2.0},
    num_results=5
)

chunkedRagModel = RAGBase(chunkedIndex, openai_client)

chunkedResponse = chunkedRagModel.rag(question)
print(response)


The agentic loop keeps calling the model in a `while True` loop. After each model response, it checks whether there were any `function_call` items:

- If there are function calls, it runs the tool, appends the tool output to the message history, and loops again.
- If there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the latest response**.


In [9]:
chunkedRagModel.cost()
# 3× fewer

Token cost:  2657


In [10]:
from agentic_rag_helper import AgenticRAGBase
from toyaikit.llm import OpenAIClient

agenticRAGmodel = AgenticRAGBase(chunkedIndex, OpenAIClient())
agenticResponse = agenticRAGmodel.agenticRAG(question)
print(response)


3
The agentic loop keeps calling the model in a `while True` loop. After each model response, it checks whether there were any `function_call` items:

- If there are function calls, it runs the tool, appends the tool output to the message history, and loops again.
- If there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the latest response**.
